# Reinforcement Learning Models of Human Decision-Making
## Multi-Armed Bandit Task: Exploration vs. Exploitation

This notebook models how humans (and computational agents) solve the **explore-exploit dilemma**:  
- Should I try a new option that might be better? (**explore**)
- Or stick with the option I know works? (**exploit**)

We simulate three agents with different strategies and fit a Q-learning model to recover parameters.

**Inspired by:** Behavioral experiments on generalization and decision-making at the Max Planck Institute for Biological Cybernetics.

---
## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from bandit_task import (
    BanditEnvironment,
    simulate_random_agent,
    simulate_greedy_agent,
    simulate_qlearning_agent,
    QLearningModel,
    plot_cumulative_rewards,
    plot_arm_selection,
    plot_parameter_recovery
)

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print('Setup complete.')

---
## 2. Define the Environment

We create a 4-armed bandit where each arm has a hidden reward probability.  
The agent doesn't know these probabilities — it has to learn them through experience.

In [ ]:
# 4 arms with known reward probabilities (simulating ground truth)
env = BanditEnvironment(
    n_arms=4,
    reward_probs=[0.2, 0.5, 0.8, 0.35],
    seed=42
)

print(env)
print(f'Best arm (highest reward prob): Arm {env.best_arm}')

---
## 3. Simulate Three Agents

We test three decision strategies:

| Agent | Strategy |
|---|---|
| **Random** | Ignores feedback, chooses randomly |
| **Greedy** | Always picks the current best option |
| **Q-Learning** | Learns gradually, balances explore/exploit |


In [ ]:
N_TRIALS = 200

# Reset environment for each agent so they face the same task
results_random  = simulate_random_agent(BanditEnvironment(4, [0.2, 0.5, 0.8, 0.35], 42), N_TRIALS)
results_greedy  = simulate_greedy_agent(BanditEnvironment(4, [0.2, 0.5, 0.8, 0.35], 42), N_TRIALS)
results_ql      = simulate_qlearning_agent(BanditEnvironment(4, [0.2, 0.5, 0.8, 0.35], 42),
                                            alpha=0.3, beta=5.0, n_trials=N_TRIALS)

# Summary statistics
summary = pd.DataFrame({
    'Agent':        ['Random', 'Greedy', 'Q-Learning'],
    'Total Reward': [
        results_random['rewards'].sum(),
        results_greedy['rewards'].sum(),
        results_ql['rewards'].sum()
    ],
    'Mean Reward':  [
        results_random['rewards'].mean().round(3),
        results_greedy['rewards'].mean().round(3),
        results_ql['rewards'].mean().round(3)
    ],
    'Optimal Arm %': [
        (results_random['choices'] == env.best_arm).mean().round(3),
        (results_greedy['choices'] == env.best_arm).mean().round(3),
        (results_ql['choices'] == env.best_arm).mean().round(3)
    ]
})

print(summary.to_string(index=False))

---
## 4. Visualize Agent Performance

In [ ]:
plot_cumulative_rewards(
    {'Random': results_random, 'Greedy': results_greedy, 'Q-Learning': results_ql},
    title='Agent Comparison: Cumulative and Running Average Reward'
)

In [ ]:
# Visualize which arm each agent chose over time
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)

for ax, (name, result) in zip(axes, [
    ('Random', results_random),
    ('Greedy', results_greedy),
    ('Q-Learning', results_ql)
]):
    ax.scatter(range(N_TRIALS), result['choices'], alpha=0.4, s=12,
               c=result['choices'], cmap='tab10', vmin=0, vmax=3)
    ax.axhline(env.best_arm, color='red', linestyle='--', alpha=0.5, label='Best arm')
    ax.set_yticks(range(4))
    ax.set_yticklabels([f'Arm {i}' for i in range(4)])
    ax.set_title(f'{name} Agent', fontsize=12)
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Trial')
plt.suptitle('Arm Selection Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Model Fitting: Recovering Parameters from Behavior

Here we do what computational neuroscientists do with real participant data:  
**Given only the choices and rewards, can we recover the agent's learning rate (α) and exploration tendency (β)?**

This is called **maximum likelihood estimation (MLE)**.

In [ ]:
# Fit the Q-learning model to the Q-learning agent's behavior
model = QLearningModel(n_arms=4)

fitted = model.fit(
    choices=results_ql['choices'],
    rewards=results_ql['rewards'],
    n_restarts=15
)

print('Model Fitting Results:')
print(f"  True α (learning rate):      0.300")
print(f"  Recovered α:                 {fitted['alpha']:.3f}")
print()
print(f"  True β (inverse temperature): 5.000")
print(f"  Recovered β:                  {fitted['beta']:.3f}")
print()
print(f"  Negative log-likelihood:      {fitted['neg_log_likelihood']:.2f}")

---
## 6. Parameter Recovery Test

A good model should recover parameters reliably across many simulated participants.  
We simulate 40 'participants' with varying α and β, then check if fitting recovers them.

In [ ]:
rng = np.random.default_rng(99)

# Simulate 40 participants with different parameters
true_alphas, true_betas = [], []
recovered_alphas, recovered_betas = [], []

n_participants = 40
print(f'Fitting {n_participants} simulated participants...')

for i in range(n_participants):
    # Random true parameters for this participant
    true_alpha = rng.uniform(0.05, 0.95)
    true_beta  = rng.uniform(0.5, 15.0)

    # Simulate their behavior
    task_env = BanditEnvironment(4, [0.2, 0.5, 0.8, 0.35], seed=i)
    result = simulate_qlearning_agent(task_env, alpha=true_alpha, beta=true_beta,
                                       n_trials=150, seed=i)

    # Fit model and recover parameters
    m = QLearningModel(n_arms=4)
    recovered = m.fit(result['choices'], result['rewards'], n_restarts=5)

    true_alphas.append(true_alpha)
    true_betas.append(true_beta)
    recovered_alphas.append(recovered['alpha'])
    recovered_betas.append(recovered['beta'])

    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{n_participants} done')

print('Done!')

In [ ]:
# Correlation between true and recovered parameters
from scipy.stats import pearsonr

r_alpha, p_alpha = pearsonr(true_alphas, recovered_alphas)
r_beta,  p_beta  = pearsonr(true_betas,  recovered_betas)

print(f'Learning rate (α):       r = {r_alpha:.3f}, p = {p_alpha:.4f}')
print(f'Inverse temperature (β): r = {r_beta:.3f},  p = {p_beta:.4f}')

plot_parameter_recovery(true_alphas, true_betas, recovered_alphas, recovered_betas)

---
## 7. Effect of Learning Rate and Temperature

How do α and β parameters shape behavior?  
Here we do a small grid search to visualize their effect on performance.

In [ ]:
alphas = [0.05, 0.2, 0.5, 0.9]
betas  = [1.0, 3.0, 7.0, 15.0]

results_grid = np.zeros((len(alphas), len(betas)))

for i, alpha in enumerate(alphas):
    for j, beta in enumerate(betas):
        total_reward = 0
        # Average over 20 runs for stability
        for run in range(20):
            task_env = BanditEnvironment(4, [0.2, 0.5, 0.8, 0.35], seed=run)
            res = simulate_qlearning_agent(task_env, alpha=alpha, beta=beta,
                                            n_trials=100, seed=run)
            total_reward += res['rewards'].mean()
        results_grid[i, j] = total_reward / 20

# Plot heatmap
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(results_grid, aspect='auto', cmap='YlOrRd',
                vmin=results_grid.min(), vmax=results_grid.max())

ax.set_xticks(range(len(betas)))
ax.set_xticklabels([f'β={b}' for b in betas])
ax.set_yticks(range(len(alphas)))
ax.set_yticklabels([f'α={a}' for a in alphas])
ax.set_xlabel('Inverse Temperature β (exploration → exploitation)', fontsize=11)
ax.set_ylabel('Learning Rate α', fontsize=11)
ax.set_title('Mean Reward by (α, β) Parameters', fontsize=13, fontweight='bold')

plt.colorbar(im, ax=ax, label='Mean reward per trial')

# Annotate cells
for i in range(len(alphas)):
    for j in range(len(betas)):
        ax.text(j, i, f'{results_grid[i, j]:.2f}', ha='center', va='center',
                fontsize=10, color='black')

plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated:

1. **Environment setup**: A 4-armed bandit with hidden reward probabilities
2. **Agent comparison**: Random < Greedy < Q-Learning in terms of performance
3. **Model fitting**: MLE recovers learning rate α and temperature β from behavioral data
4. **Parameter recovery**: The model reliably recovers true parameters across simulated participants
5. **Parameter sensitivity**: Intermediate α and high β tend to maximize reward

### Relevance to real behavioral research:
- In human experiments, we observe **choices and rewards only** — we don't know the true parameters
- Fitting this model to real data reveals individual differences in learning rate and exploration
- These parameters correlate with age, psychiatric conditions, and brain activity (e.g., dopamine)

---
*Author: Selin Doğaner | github.com/selindgnr*